# Wine Cultivar Classification
### Comparing Classical ML Models on Chemical Analysis Data

This notebook walks through a complete, lightweight ML workflow: EDA, preprocessing, model comparison via cross-validation, hyperparameter tuning, and evaluation — all on a small tabular dataset that runs in seconds on a CPU (no GPU, no internet download needed).

**Dataset**: [UCI Wine Dataset](https://archive.ics.uci.edu/dataset/109/wine) (bundled with scikit-learn) — results of a chemical analysis of 178 wines grown in the same region of Italy, derived from three different cultivars (grape varieties). 13 numeric features: alcohol content, malic acid, ash, flavanoids, color intensity, proline, etc.

In [ ]:
import sys
sys.path.append('..')

from src.data_prep import load_data, prepare_splits
from src.train_models import compare_models, get_candidate_models, tune_model, PARAM_GRIDS
from src.evaluate import evaluate_model, plot_feature_importance
from src.utils import set_seed, plot_class_balance, plot_correlation_heatmap, plot_pca_projection

set_seed(42)

## 1. Load & explore the data

In [ ]:
X, y, feature_names, target_names = load_data()
print('Shape:', X.shape)
print('Classes:', target_names)
X.head()

In [ ]:
X.describe()

In [ ]:
plot_class_balance(y, target_names, '../results/class_balance.png')

In [ ]:
plot_correlation_heatmap(X, '../results/correlation_heatmap.png')

In [ ]:
# PCA projection: are the 3 cultivars actually separable in feature space?
plot_pca_projection(X.values, y.values, target_names, '../results/pca_projection.png')

## 2. Train/test split + feature scaling

In [ ]:
X_train, X_test, y_train, y_test, scaler = prepare_splits(X, y)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 3. Compare candidate models (5-fold cross-validation)

We compare four classical algorithms before committing to one, rather than jumping straight to the fanciest model. This is good practice: simpler models (like Logistic Regression) are often competitive on small, clean, well-scaled datasets like this one.

In [ ]:
cv_results = compare_models(X_train, y_train)

## 4. Hyperparameter tuning of the best candidate

In [ ]:
best_model_name = max(cv_results, key=lambda k: cv_results[k]['mean_accuracy'])
print('Best model from CV:', best_model_name)

base_model = get_candidate_models()[best_model_name]
best_model, best_params, best_cv_score = tune_model(
    base_model, PARAM_GRIDS[best_model_name], X_train, y_train
)

## 5. Evaluate on the held-out test set

In [ ]:
y_pred = evaluate_model(best_model, X_test, y_test, target_names, results_dir='../results')

In [ ]:
plot_feature_importance(
    best_model, feature_names, '../results/feature_importance.png',
    X_test=X_test, y_test=y_test,
)

## 6. Conclusions

- All four models scored above ~95% cross-validated accuracy, which suggests the three cultivars are close to linearly/kernel-separable given these 13 chemical features — confirmed visually by the PCA plot in section 1.
- The tuned model reaches strong test-set performance; see the confusion matrix and classification report above for the per-class breakdown.
- Permutation/feature importance highlights which chemical properties (e.g. flavanoids, color intensity, proline) are most informative for distinguishing cultivars — useful if this were to inform an actual chemistry-based sorting process.
- **Limitations**: only 178 samples total and a single wine-growing region, so generalization to other regions/vintages is untested. A next step would be testing on an external wine dataset.